In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

In [127]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [4]:
df=pd.read_csv('karachi_properties_outlier_treated.csv')

In [5]:
df.shape

(20479, 29)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20479 entries, 0 to 20478
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   property_name       20479 non-null  object 
 1   address             20479 non-null  object 
 2   location            20479 non-null  object 
 3   sub_location        18297 non-null  object 
 4   price               20479 non-null  float64
 5   price_crore         20479 non-null  float64
 6   price_per_sqft      20479 non-null  float64
 7   area                20479 non-null  float64
 8   bedrooms            20479 non-null  int64  
 9   bathrooms           20479 non-null  int64  
 10  kitchen             20447 non-null  float64
 11  floor               3473 non-null   float64
 12  total_floors        9617 non-null   float64
 13  parking             20385 non-null  float64
 14  elevator            20479 non-null  int64  
 15  servant_quarters    20479 non-null  float64
 16  elec

In [7]:
df[df['sub_location'].isnull()]['property_name'].value_counts().index

Index(['flat for sale', 'house for sale',
       'leased bank loan applicable brand new flat also available for sale',
       '1 bed lounge apartment for sale | ready to move | opposite fast university | national highway',
       'brand new apartment available for sale',
       'apartment available for sale', 'apartment for sale',
       'brand new house for sale', 'brand new bungalow for sale',
       'bungalow for sale',
       ...
       'bank loan not applicable 2 bed dd apartment for sale in mateen complex | main national highway | apartments for sale',
       'two unit bungalow available for sale',
       'remco tower 4bed drawing for sale ultra luxury flat. modern interior',
       'prime location house of 1250 square feet in naya nazimabad for sale',
       'brand new town house for sale',
       'flat for sale in sawera enclave (a project of ok estate & builders)',
       'get in touch now to buy a flat in gulshan-e-iqbal',
       'prime location 1180 square feet spacious flat

In [8]:
for item in df[df['sub_location'].isnull()]['property_name'].value_counts().index.tolist():
    print(item)

flat for sale
house for sale
leased bank loan applicable brand new flat also available for sale
1 bed lounge apartment for sale | ready to move | opposite fast university | national highway
brand new apartment available for sale
apartment available for sale
apartment for sale
brand new house for sale
brand new bungalow for sale
bungalow for sale
house available for sale
2 bed lounge apartment for sale in mateen complex ready to move main national highway
brand new flat is for sale
located in the heart of scheme 33 gulshan-e-maymar, rainbow twin tower
prime location 1250 square feet flat for sale in naya nazimabad
brand new houses for sale
leased bank loan applicable brand new flat available for sale
beautiful apartment available for sale
prime location 120 square yards house for sale in naya nazimabad
prime location flat for sale
(direct owner deal)
brand new house available for sale
beautiful apartment for sale
brand new project burj-ul-harmain at main safoora chowrangi
flat available

In [9]:
df['sub_location'].value_counts()

sub_location
Phase 8                                         2049
Phase 6                                          884
Askari 5                                         498
Askari 6                                         489
Falcon Complex New Malir                         449
Gulshan-E-Maymar                                 436
Gulshan-e-Iqbal Block 10A                        422
Phase 5                                          421
Sector 35-A                                      301
Phase 7                                          286
Pechs                                            268
Sector 47                                        260
Sector 40                                        235
Falaknaz                                         220
Surjani Town                                     209
Gulistan-e-Jauhar Block 15                       186
North Nazimabad Block F                          179
Shaheed Millat Road                              173
Saadi Garden                     

In [10]:
import re



sub_locations = (
    df['sub_location']
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)


sub_locations = sorted(sub_locations, key=len, reverse=True)


def extract_sub_location(property_name):
    if pd.isna(property_name):
        return np.nan

    property_name = str(property_name)

    for sub_loc in sub_locations:

        pattern = re.escape(sub_loc)


        if re.search(pattern, property_name, flags=re.IGNORECASE):
            return sub_loc

    return np.nan



mask = df['sub_location'].isna()

df.loc[mask, 'sub_location'] = (
    df.loc[mask, 'property_name']
      .apply(extract_sub_location)
)

In [11]:
df['sub_location'].isnull().sum()

np.int64(2009)

In [12]:
import re
import numpy as np
import pandas as pd

# Existing known sub-locations
sub_locations = (
    df['sub_location']
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

# Match longer names first
sub_locations = sorted(sub_locations, key=len, reverse=True)


def extract_sub_location(property_name):
    if pd.isna(property_name):
        return np.nan

    text = str(property_name)

    
    for sub_loc in sub_locations:
        pattern = re.escape(sub_loc)

        if re.search(pattern, text, flags=re.IGNORECASE):
            return sub_loc

    # --------------------------------------------------
    # 2. Extract Phase
    # Examples:
    # Phase 8
    # Phase 8 Extension
    # Phase 2
    # Phase 2 Extension
    # --------------------------------------------------
    match = re.search(
        r'\bPhase\s+\d+(?:\s+Extension)?\b',
        text,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(0).strip()

    # --------------------------------------------------
    # 3. Extract Sector
    # Examples:
    # Sector 35-A
    # Sector 11
    # Sector 13D
    # Sector 24-B
    # --------------------------------------------------
    match = re.search(
        r'\bSector\s+\d+[A-Za-z]?(?:[-/][A-Za-z0-9]+)?\b',
        text,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(0).strip()

    # --------------------------------------------------
    # 4. Extract Block
    # Examples:
    # Block F
    # Block 10A
    # Block 13D2
    # Block A
    # --------------------------------------------------
    match = re.search(
        r'\bBlock\s+[A-Za-z0-9]+(?:[-/][A-Za-z0-9]+)?\b',
        text,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(0).strip()

    return np.nan


# Only fill missing sub_location
mask = df['sub_location'].isna()

df.loc[mask, 'sub_location'] = (
    df.loc[mask, 'property_name']
      .apply(extract_sub_location)
)

In [13]:
df.groupby('location')['sub_location'].value_counts()

location              sub_location                                
Bahria Town           Bahria Sports City                                94
                      Precinct 2                                        92
                      Precinct 12                                       85
                      Precinct 8                                        75
                      Bahria Apartments                                 74
                      Private High-Rises                                70
                      Precinct 11A                                      58
                      Bahria Paradise                                   56
                      Precinct 1                                        54
                      Precinct 31                                       54
                      Precinct 9                                        54
                      Precinct 10A                                      50
                      Precinct 11

In [14]:
df.loc[
    df['sub_location'].eq('Mateen Complex'),
    'location'
] = 'Bin Qasim Town'

In [15]:
import re

mask = (
    (df['location'] == 'Gulshan-e-Iqbal Town') &
    (df['sub_location'].str.match(r'^\s*block\s+', case=False, na=False))
)

df.loc[mask, 'sub_location'] = (
    df.loc[mask, 'sub_location']
      .str.strip()
      .str.replace(r'^block\s+', 'Gulshan-e-Iqbal Block ', regex=True, case=False)
)

In [16]:
gulshan_mapping = {
    'Gulshan-e-Iqbal Block 10a': 'Gulshan-e-Iqbal Block 10A',
    'Gulshan-e-Iqbal Block 13d2': 'Gulshan-e-Iqbal Block 13D2',
    'Gulshan-e-Iqbal Block 13d': 'Gulshan-e-Iqbal Block 13D',
    'Gulshan-e-Iqbal Block 15': 'Gulshan-e-Iqbal Block 15',
    'Gulshan-e-Iqbal Block 18': 'Gulshan-e-Iqbal Block 18',
    'Gulshan-e-Iqbal Block 13a': 'Gulshan-e-Iqbal Block 13A',
}

df['sub_location'] = df['sub_location'].replace(gulshan_mapping)

In [17]:
invalid_sub_locations = [
    'Gulshan-e-Iqbal Block a',
    'Gulshan-e-Iqbal Block c',
    'Gulshan-e-Iqbal Block of'
]

df = df[~df['sub_location'].isin(invalid_sub_locations)].copy()

In [18]:
df.loc[
    df['sub_location'].eq('Hill Park'),
    'sub_location'
] = 'Jamshed Town'

In [19]:
federal_b_area_mapping = {
    'block 5': 'Federal B Area Block 5',
    'block 4': 'Federal B Area Block 4',
    'block 18': 'Federal B Area Block 18',
    'block 3': 'Federal B Area Block 3',
    'block 11': 'Federal B Area Block 11',
    'block 17': 'Federal B Area Block 17',
    'block 2': 'Federal B Area Block 2',
    'block 21': 'Federal B Area Block 21',
    'block 9': 'Federal B Area Block 9',
}

mask = df['location'].eq('Federal B Area')

df.loc[mask, 'sub_location'] = (
    df.loc[mask, 'sub_location']
      .replace(federal_b_area_mapping)
)

In [20]:
gulistan_jauhar_mapping = {
    'block 13': 'Gulistan-e-Jauhar Block 13',
    'block 2': 'Gulistan-e-Jauhar Block 2',
    'block 15': 'Gulistan-e-Jauhar Block 15',
    'block 16': 'Gulistan-e-Jauhar Block 16',
    'block 19': 'Gulistan-e-Jauhar Block 19',
    'block 3a': 'Gulistan-e-Jauhar Block 3A',
    'block 7': 'Gulistan-e-Jauhar Block 7',
    'block 12': 'Gulistan-e-Jauhar Block 12',
    'block 4': 'Gulistan-e-Jauhar Block 4',
    'block 10': 'Gulistan-e-Jauhar Block 10',
    'block 14': 'Gulistan-e-Jauhar Block 14',
    'block 3': 'Gulistan-e-Jauhar Block 3',
    'block 5': 'Gulistan-e-Jauhar Block 5',
    'block 02': 'Gulistan-e-Jauhar Block 2',
    'block 03-a': 'Gulistan-e-Jauhar Block 3A',
    'block 03a': 'Gulistan-e-Jauhar Block 3A',
    'block 04': 'Gulistan-e-Jauhar Block 4',
    'block 05': 'Gulistan-e-Jauhar Block 5',
    'block 1': 'Gulistan-e-Jauhar Block 1',
    'block 10a': 'Gulistan-e-Jauhar Block 10',
    'block 11': 'Gulistan-e-Jauhar Block 11',
    'block 16-a': 'Gulistan-e-Jauhar Block 16A',
    'block 16a': 'Gulistan-e-Jauhar Block 16A',
    'block 17': 'Gulistan-e-Jauhar Block 17',
    'block 9a': 'Gulistan-e-Jauhar Block 9',
}

mask = df['location'].eq('Gulistan-e-Jauhar')

df.loc[mask, 'sub_location'] = (
    df.loc[mask, 'sub_location']
      .replace(gulistan_jauhar_mapping)
)

In [21]:
mask = df['sub_location'].str.strip().str.lower().eq('airport')

df.loc[mask, 'location'] = 'Malir'

In [22]:
# Map valid lowercase blocks
naya_nazimabad_mapping = {
    'block a': 'Naya Nazimabad Block A',
    'block c': 'Naya Nazimabad Block C',
    'block d': 'Naya Nazimabad Block D',
    'block m': 'Naya Nazimabad Block M',
}

mask = df['location'].eq('Naya Nazimabad')

df.loc[mask, 'sub_location'] = (
    df.loc[mask, 'sub_location']
      .replace(naya_nazimabad_mapping)
)


# Delete invalid "block house"
df = df[
    ~(
        df['location'].eq('Naya Nazimabad') &
        df['sub_location'].str.strip().str.lower().eq('block house')
    )
].copy()

In [23]:
df[(df['location'] == 'Bahria Town') & (df['sub_location'].isnull())][
    'property_name']

322      152 square yards villa available for sale in b...
399      850 square feet's apartments available for sal...
442                                      bungalow for sale
549      1500 square feet ready to move apartments avai...
1284     grey structure villa for sale in bahria town k...
1596     850 square feet 2 bedroom flat available for s...
2643     950 square feet flat available for sale in bah...
2718     brand new 500 yards villa chance deal best pri...
3215     bahria hills, p9-500sq yds, most prime locatio...
3463     3 bedroom grey structure villa at prime locati...
3553     bahria hills 500 sq yards double story with ba...
4001     fully renovated villa for sale in paradise bahria
4350     200 yds one unit prime location villa at muree...
4507     most prime apartment of bahria town karachi sp...
4774     1000sq yards, precent 7, brandnew banglow, top...
4810     350 yard ready to move brand villa available s...
5161     2 bed lounge flat available on booking in us &.

In [24]:
# Save the filtered property names to a CSV file
df[(df['location'] == 'Bahria Town') & (df['sub_location'].isnull())][
    'property_name'
].to_csv('bahria_town_null_sublocations.csv', index=False)

In [25]:
# Save the value_counts result to an Excel file
df[df["location"] == "Bahria Town"]["sub_location"].value_counts()

sub_location
Bahria Sports City    94
Precinct 2            92
Precinct 12           85
Precinct 8            75
Bahria Apartments     74
Private High-Rises    70
Precinct 11A          58
Bahria Paradise       56
Precinct 1            54
Precinct 9            54
Precinct 31           54
Precinct 10A          50
Precinct 11B          48
Bahria Heights        47
Precinct 6            45
Precinct 10B          42
Precinct 27           40
Precinct 4            37
Precinct 19           32
Precinct 16           30
Precinct 17           19
Precinct 15           17
Precinct 7             7
Precinct 18            5
Name: count, dtype: int64

In [26]:
sub_location_mapping = {
    3215: 'Precinct 9',        
    3553: 'Precinct 9',        
    4001: 'Bahria Paradise',   
    4774: 'Precinct 7',        
    18993: 'Precinct 22',      
    20403: 'Precinct 9',       
    20431: 'Precinct 20',      
}

In [27]:
df[(df['location'] == 'Bin Qasim Town') & (df['sub_location'].isnull())][
    'property_name']

590      newly constructed park face house for sale adj...
6001                                        house for sale
6228                              double road corner house
8508                                     bungalow for sale
10481              flat for sale ground floor 2 bed lounge
10503    1 bed lounge apartment for sale | ready to mov...
10770    highly-desirable 120 square yards house availa...
11519    buy a house of 180 square yards in pn welfare ...
12096    well designed and renovated house in safe and ...
13619    1 bed lounge apartment for sale | ready to mov...
14789    1 bed lounge apartment for sale | ready to mov...
16634    1 bed lounge apartment for sale | ready to mov...
Name: property_name, dtype: object

In [28]:
df[(df['location'] == 'Bin Qasim Town') & (df['sub_location'].isnull())][
    'property_name'
].to_csv('bahria_town_null_sublocations.csv', index=False)

In [29]:
df[df["location"] == "Bin Qasim Town"]["sub_location"].value_counts()

sub_location
Mateen Complex    35
Phase 2            2
Model Colony       2
block a            1
Name: count, dtype: int64

In [30]:
df = df[~((df['location'] == 'Bin Qasim Town') & (df['sub_location'].isnull()))]

In [31]:
df[(df['location'] == 'Bin Qasim Town ') & (df['sub_location'].isnull())][
    'property_name']

Series([], Name: property_name, dtype: object)

In [32]:
mask = df['location'].eq('Bin Qasim Town') & df['sub_location'].isin(['Model Colony', 'Phase 2', 'block a'])

df.loc[mask, 'sub_location'] = None

In [33]:
df = df[~((df['location'] == 'Bin Qasim Town') & (df['sub_location'].isnull()))]

In [34]:
df[(df['location'] == 'Clifton') & (df['sub_location'].isnull())][
    'property_name']

249          flat for sale in clifton ideal family living!
812                   fresh booking flat available clifton
900      prime location in punjab chowrangi 1600 square...
1383     2628 square feet flat is available in grand mo...
1459                              brand new apartment sale
1680                          apartment available for sale
1972     a palatial residence for prime location sale i...
2034                brand new apartment available for sale
2258         brand new duplex apartment available for sale
2598                              flat for sale in karachi
3067          flat of 1800 square feet in clifton for sale
3727     1800 square feet apartment available for sale ...
5900     luxurious sea facing 3 bedrooms apartment for ...
7283     spacious 3 bedrooms apartment for sale servant...
7318                             brand new duplex for sale
7563                brand new apartment available for sale
7664                          apartment available for sa

In [35]:
df[(df['location'] == 'Clifton') & (df['sub_location'].isnull())][
    'property_name'
].to_csv('Clifton_null_sublocations.csv', index=False)

In [36]:
mask = df['location'].eq('DHA City')

df.loc[mask, 'location'] = 'Super Highway'
df.loc[mask, 'sub_location'] = 'DHA City'

In [37]:
df[(df['location'] == 'DHA Defence') & (df['sub_location'].isnull())][
    'property_name'
].to_csv('DHA_Defence_null_sublocations.csv', index=False)

In [38]:
df[(df['location'] == 'DHA Defence') & (df['sub_location'].isnull())][
    'property_name']

24       prime location 2700 square feet flat for sale ...
104      huge driveway offering ample parking space hou...
432      prime location flat of 2350 square feet in sea...
564                                      apartment on sale
842      unparalleled luxury living at nhs zamzama kara...
883                  sea view apartment available for sale
1165     fully renovated seaview apartment second floor...
1222     seaview apartment first floor ff-2 available f...
1392     fully renovated sea view apartment 2nd floor sv 5
1597        furnished seaview apartment available for sale
1744     fully renovated seaview apartment ground floor...
2353     seaview apartment second floor available for sale
2397     sefacing luxury sea view apartment second floo...
2501                                      galaxy sky tower
2808                          sea view apartment for sale.
2952       affordable flat for sale in sea view apartments
3328                  renovated seaview apartment for sa

In [39]:
sea_view_apartment_indexes = [
    883,
    1165,
    1222,
    1392,
    1597,
    1744,
    2353,
    2397,
    2808,
    2952,
    3328,
    3888,
    4379,
    6717,
    8418,
    8477,
    8536,
    8952,
    11099,
    11910,
    12837,
    13893,
    15084,
    15766,
    20040,
    20053,
    20055
]

df.loc[
    df.index.isin(sea_view_apartment_indexes),
    'sub_location'
] = 'Phase 5'

In [40]:
df[(df['location'] == 'Malir') & (df['sub_location'].isnull())][
    'property_name']

261      3 bed lounge flat next to corner jinnah avenue...
288      tipu sultan housing society malir cantt karach...
365      flat for booking in sumsum grand 2 on main jin...
433      80 sq. yards corner house + road facing shop f...
1558           apartment for sale on easy installment plan
1575     08 bed dd g+2 bungalow for sale in tipu sultan...
3206          flat for grabs in 70 square yards moria goth
3208     chance deal ground +2 house for sale in malir ...
3667     prime location 160 square yards house in kazim...
3973                           gohar complex flat for sale
4109     brand new flat for sale at al amin city malir ...
4548                             new project flat for sale
4656     this is the most luxurious, fully maintained, ...
5094     brand new house 240 sq leased map approved for...
5613                            3 bed lounge flat for sale
6314     gohar complex brand new 3 sides corner, west o...
6779     tipu sultan housing society near malir cantt c.

In [41]:
df[(df['location'] == 'Malir') & (df['sub_location'].isnull())][
    'property_name'
].to_csv('Malir Town_null_sublocations.csv', index=False)

In [42]:
mateen_complex_idx = [14603, 15060, 15474, 19421]

df.loc[
    df.index.isin(mateen_complex_idx),
    'location'
] = 'Bin Qasim Town'

df.loc[
    df.index.isin(mateen_complex_idx),
    'sub_location'
] = 'Mateen Complex'


shahra_e_faisal_idx = [7968, 13129, 13949]

df.loc[
    df.index.isin(shahra_e_faisal_idx),
    'location'
] = 'Shahra-e-Faisal'

In [43]:
df[(df['location'] == 'Malir') & (df['sub_location'].isnull())][
    'property_name']

261      3 bed lounge flat next to corner jinnah avenue...
288      tipu sultan housing society malir cantt karach...
365      flat for booking in sumsum grand 2 on main jin...
433      80 sq. yards corner house + road facing shop f...
1558           apartment for sale on easy installment plan
1575     08 bed dd g+2 bungalow for sale in tipu sultan...
3206          flat for grabs in 70 square yards moria goth
3208     chance deal ground +2 house for sale in malir ...
3667     prime location 160 square yards house in kazim...
3973                           gohar complex flat for sale
4109     brand new flat for sale at al amin city malir ...
4548                             new project flat for sale
4656     this is the most luxurious, fully maintained, ...
5094     brand new house 240 sq leased map approved for...
5613                            3 bed lounge flat for sale
6314     gohar complex brand new 3 sides corner, west o...
6779     tipu sultan housing society near malir cantt c.

In [44]:
# Nazimabad 1, 2, 3 → Nazimabad
mask = (
    (df['location'] == 'Naya Nazimabad') &
    (df['sub_location'].isin(['Nazimabad 1', 'Nazimabad 2', 'Nazimabad 3']))
)

df.loc[mask, 'location'] = 'Nazimabad'

In [45]:
df[(df['location'] == 'Scheme 33') & (df['sub_location'].isnull())][
    'property_name']

18       prime location sale the ideally located house ...
121                 scheme 33 apartment available for sale
131      sawera grand enclave (a project by ok estate &...
166      a stunning house is up for grabs in scheme 33 ...
172                      roomi boulevard new project louch
225      brand new flat for buy in scheme 33 near madra...
274      road facing 2 bed dd flat with roof for sale g...
444      modern living redefined apartments for sale in...
497             flat available on booking in saima project
553             pakistan air crew bondrt wall society west
687                   transfer house is available for sale
697                                          flat for sale
709      pcsir society scheme-33 240square yards single...
725      prime location 945 square feet flat in scheme ...
764      3 bed dd apartment available for sell in sakin...
771      end your search for prime location flat here a...
824      a 120 square yards house has landed on market .

In [46]:
df[(df['location'] == 'Scheme 33') & (df['sub_location'].isnull())][
    'property_name'
].to_csv('Scheme 33_null_sublocations.csv', index=True)

In [47]:
# Map original dataframe index -> sub_location
# Only fill rows where sub_location is currently NaN

sector_mapping = {
    # Sector 29
    131: "Sector 29",
    444: "Sector 29",
    1316: "Sector 29",
    1354: "Sector 29",
    1384: "Sector 29",
    1480: "Sector 29",
    2229: "Sector 29",
    2474: "Sector 29",
    2507: "Sector 29",
    2706: "Sector 29",
    3270: "Sector 29",
    4675: "Sector 29",
    5018: "Sector 29",
    5392: "Sector 29",
    5672: "Sector 29",
    6224: "Sector 29",
    6639: "Sector 29",
    7162: "Sector 29",
    7769: "Sector 29",
    7817: "Sector 29",
    8038: "Sector 29",
    8124: "Sector 29",
    8251: "Sector 29",
    8458: "Sector 29",
    9049: "Sector 29",
    9842: "Sector 29",
    11901: "Sector 29",
    12603: "Sector 29",
    13826: "Sector 29",
    14235: "Sector 29",
    14664: "Sector 29",
    15825: "Sector 29",
    16756: "Sector 29",
    17019: "Sector 29",
    17036: "Sector 29",
    17191: "Sector 29",
    17629: "Sector 29",
    18000: "Sector 29",
    18047: "Sector 29",
    18058: "Sector 29",
    18420: "Sector 29",
    20046: "Sector 29",
    20249: "Sector 29",

    # Sector 30
    497: "Sector 30",
    16498: "Sector 30",

    # Sector 18-A
    1483: "Sector 18-A",

    # Sector 17-A
    7978: "Sector 17-A",
    9384: "Sector 17-A",
    11051: "Sector 17-A",
    11223: "Sector 17-A",

    # Sector 24-A
    2645: "Sector 24-A",
    3073: "Sector 24-A",
    5192: "Sector 24-A",
    5360: "Sector 24-A",
    7387: "Sector 24-A",
    14449: "Sector 24-A",
    15126: "Sector 24-A",
    18735: "Sector 24-A",

    # Sector 35-A
    764: "Sector 35-A",
    4645: "Sector 35-A",
    5209: "Sector 35-A",
    15602: "Sector 35-A",

    # Sector 42-A
    3676: "Sector 42-A",

    # Sector 40
    3997: "Sector 40",
    4082: "Sector 40",
    4103: "Sector 40",
    5032: "Sector 40",
    10694: "Sector 40",
    12554: "Sector 40",
    12705: "Sector 40",
    12708: "Sector 40",
    16949: "Sector 40",
    17198: "Sector 40",
    18955: "Sector 40",
    19264: "Sector 40",
    19277: "Sector 40",
    19286: "Sector 40",
    19670: "Sector 40",
    19698: "Sector 40",

    # Sector 19-A
    3680: "Sector 19-A",
    10285: "Sector 19-A",
    12340: "Sector 19-A",
    15054: "Sector 19-A",
    15859: "Sector 19-A",
    15900: "Sector 19-A",
    17057: "Sector 19-A",
    18080: "Sector 19-A",

    # Sector 34-A
    6985: "Sector 34-A",
    14069: "Sector 34-A",
    14758: "Sector 34-A",

    # Sector 15-A
    12372: "Sector 15-A",
    19309: "Sector 15-A",

    # Sector 4-C
    11637: "Sector 4-C",

    # Sector 4-A
    12084: "Sector 4-A",

    # Sector 25-B
    5897: "Sector 25-B",
    12889: "Sector 25-B",

    # Sector 33
    14551: "Sector 33",
}


# Only fill NaN values in the EXISTING sub_location column
for idx, sector in sector_mapping.items():
    if idx in df.index and pd.isna(df.loc[idx, "sub_location"]):
        df.loc[idx, "sub_location"] = sector

In [48]:
sector_normalization = {
    "sector 21-a": "Sector 21-A",
    "sector 13-a": "Sector 13-A",
    "sector 17a": "Sector 17-A",
    "sector 25a": "Sector 25-A",
    "sector 34a": "Sector 34-A",
    "sector 5a": "Sector 5-A",
    "sector 9-a": "Sector 9-A",
    "sector 9a/1": "Sector 9-A",
    
    "block 1": "Block 1",
    "block 2": "Block 2",
    "block 9": "Block 9",
    
    "phase 1": "Phase 1",
    "phase 3": "Phase 3",
}

df["sub_location"] = df["sub_location"].replace(sector_normalization)

In [49]:
df['sub_location'].isnull().sum()

np.int64(1586)

In [50]:
df[(df['location'] == 'Super Highway') & (df['sub_location'].isnull())][
    'property_name']

1419                    sawera grand enclave flat for sale
3096          2 shops + house main road facing chance deal
5722     legacy icon prime location flat for sale in ka...
7767     1350 square feet flat in central karachi - hyd...
9810     introducing a brand new premium project on sup...
10076    flat available for booking in sawera grand enc...
10977    bismillah center 2 bed lounge super highway ne...
11111                                       prime location
11334    flat available for booking in sawera grand enc...
11411    get in touch now to buy a prime location flat ...
11667                            beautifull villa for sale
12339    prime location in karachi - hyderabad motorway...
12347    legacy icon your address to prime location kar...
12642    luxury brand new west open bungalow for sale i...
12969    flat available for booking in sawera grand enc...
13846    ideal prime location flat for sale in karachi ...
15663    legacy icon fair-priced prime location in kara.

In [51]:
df[df['sub_location'].isnull()]['location'].value_counts()

location
Naya Nazimabad          300
Scheme 33               282
Gulshan-e-Iqbal Town    178
Gulistan-e-Jauhar        95
Malir Cantt              91
Civil Lines              79
Saddar                   69
Bahria Town              54
North Nazimabad          53
Clifton                  51
North Karachi            46
Federal B Area           45
Defence View             40
Malir                    34
Shahra-e-Faisal          32
Tipu Sultan Road         32
DHA Defence              26
Northern Bypass          26
Super Highway            23
Shah Faisal              13
Jamshed Town              7
Nazimabad                 7
Korangi                   3
Name: count, dtype: int64

In [52]:
df[(df['location'] == 'Naya Nazimabad') & (df['sub_location'].isnull())][
    'property_name']

161      prime location affordable house for sale in na...
233      a prime location 1750 square feet house has la...
289      get this amazing prime location 1950 square fe...
303      1280 square feet flat situated in naya nazimab...
320      ideal prime location 160 square yards house av...
414      buy your ideal facing park 120 square yards ho...
438      naya nazimabad peace apartment | 6.5 years eas...
486      prime location 120 square yards house in naya ...
598      prime location affordable house for sale in na...
602      2&3 bedrooms apartment | naya nazimabad | glob...
609                         naya nazimabad peace apartment
629      naya nazimabad luxurious apartment fl-05 | eas...
639      prime location naya nazimabad flat sized 1250 ...
652      prime location property for sale in naya nazim...
659      naya nazimabad: secure, boundary-walled apartm...
960      buy prime location 1750 square feet house at h...
1011     prime location flat for sale in naya nazimabad.

In [53]:
df[(df['location'] == 'Naya Nazimabad') & (df['sub_location'].isnull())][
    'property_name'
].to_csv('Naya Nazimabad_null_sublocations.csv', index=True)

In [54]:
sublocation_count = (
    df.groupby(["location", "sub_location"])["sub_location"]
      .transform("count")
)
mask = (
    df["sub_location"].isna() |
    (sublocation_count < 5)
)

df.loc[mask, "sub_location"] = df.loc[mask, "location"]

In [55]:
df['sub_location'].isnull().sum()

np.int64(0)

In [56]:
df['location'].value_counts()

location
DHA Defence             4210
Scheme 33               3063
Gulshan-e-Iqbal Town    2205
Gulistan-e-Jauhar       1748
Bahria Town             1239
Malir Cantt             1135
North Nazimabad         1020
Jamshed Town             980
Malir                    922
Clifton                  815
Gadap                    694
Naya Nazimabad           627
Federal B Area           425
North Karachi            379
Super Highway            161
Shah Faisal              152
Korangi                  151
Karsaz                   124
Nazimabad                 82
Civil Lines               79
Saddar                    69
Defence View              47
Bin Qasim Town            39
Shahra-e-Faisal           34
Tipu Sultan Road          32
Northern Bypass           26
Name: count, dtype: int64

In [57]:
df['location_score'].isnull().sum()

np.int64(348)

In [58]:
df[df['location_score'].isnull()]['location'].value_counts()

location
Civil Lines         79
Saddar              69
Super Highway       54
Defence View        47
Tipu Sultan Road    32
Shahra-e-Faisal     31
Northern Bypass     26
Malir               10
Name: count, dtype: int64

In [66]:
location_score = {
    # 5 - Premium / Highly Desirable
    "DHA Defence": 5,
    "Clifton": 5,
    "Karsaz": 5,
    "Malir Cantt": 5,

    # 4 - High Demand / Established
    "Jamshed Town": 4,
    "Gulshan-e-Iqbal Town": 4,
    "Civil Lines": 4,
    "North Nazimabad": 4,
    "Federal B Area": 4,
    "Malir": 4,
    "Naya Nazimabad": 4,
    "Shahra-e-Faisal": 4,

    # 3 - Good Residential / Established
    "Gulistan-e-Jauhar": 3,
    "Scheme 33": 3,
    "Bahria Town": 3,
    "Nazimabad": 3,
    "Saddar": 3,
    "Shah Faisal": 3,

    # 2 - Developing / Moderate
    "Northern Bypass": 2,
    "North Karachi": 2,
    "Gadap": 2,
    "Super Highway": 2,

    # 1 - Lower Residential Desirability
    "Korangi": 1,
    "Bin Qasim Town": 1
}



df["location_score"] = df["location"].map(location_score)
df.groupby("location_score")["price_per_sqft"].agg(["count", "mean", "median"])

,count,mean,median
location_score,,,
1,190,8095.816158,5555.56
2,1260,14961.879167,13130.32
3,6353,16228.628102,13777.78
4,6371,25741.603562,23611.11
5,6284,35112.914086,32532.53


In [60]:
df.groupby('location')['price_crore'].agg(['mean','median']).sort_values(by='mean',ascending=False)

,mean,median
location,,
Karsaz,21.504032,20.875
DHA Defence,15.325057,11.000
Tipu Sultan Road,11.398438,8.100
Jamshed Town,8.308224,5.500
Northern Bypass,8.050000,6.785
Gulshan-e-Iqbal Town,7.479763,4.000
Civil Lines,7.418228,8.250
Clifton,6.921080,5.850
Malir Cantt,6.689782,5.350


In [61]:
df.loc[df["location"] == "Tipu Sultan Road", "sub_location"] = "Tipu Sultan Road"
df.loc[df["sub_location"] == "Tipu Sultan Road", "location"] = "Jamshed Town"

In [62]:
same = df["location"].eq(df["sub_location"])

df.loc[same, "sub_location"] = (
    df.loc[same, "location"] + "_Unspecified"
)

In [63]:
mask = df["location"].eq("Defence View")

df.loc[mask, "location"] = "Jamshed Town"
df.loc[mask, "sub_location"] = "Defence View"

In [64]:
df[df['location']=='Jamshed Town']['sub_location'].value_counts()

sub_location
Pechs                                           268
Shaheed Millat Road                             173
Khalid Bin Walid Road                           121
Jamshed Road                                     60
Defence View                                     47
Smchs - Sindhi Muslim Society                    43
Karachi Administration Employees Society         41
Tariq Road                                       35
Amir Khusro                                      33
Jamshed Town_Unspecified                         32
Tipu Sultan Road                                 32
Garden East                                      31
Amil Colony                                      30
Soldier Bazar                                    20
Parsi Colony                                     20
Mehmoodabad                                      18
Cosmopolitan Society                             17
Shahra-E-Qaideen                                 14
Muslimabad Society                               12

In [68]:
import pandas as pd

# ============================================================
# Step 1: Calculate median price/sqft for each sub-location
# ============================================================

sub_stats = (
    df.groupby("sub_location")
      .agg(
          median_ppsqft=("price_per_sqft", "median"),
          count=("price_per_sqft", "size")
      )
      .reset_index()
)


# ============================================================
# Step 2: Calculate global percentile of sub-locations
# ============================================================

MIN_COUNT = 10

valid = sub_stats["count"] >= MIN_COUNT

sub_stats.loc[valid, "global_percentile"] = (
    sub_stats.loc[valid, "median_ppsqft"]
    .rank(method="average", pct=True)
)


# ============================================================
# Handle rare sub-locations
# ============================================================

median_pct = (
    sub_stats.loc[valid, "global_percentile"]
    .median()
)

sub_stats["global_percentile"] = (
    sub_stats["global_percentile"]
    .fillna(median_pct)
)


# ============================================================
# Step 3: Map percentile back to main dataframe
# ============================================================

pct_map = (
    sub_stats
    .set_index("sub_location")["global_percentile"]
)

df["global_percentile"] = (
    df["sub_location"].map(pct_map)
)


# ============================================================
# Step 4: Normalize parent location score
# 1–5 → 0–1
# ============================================================

parent_norm = (
    (df["location_score"] - 1) / 4
)


# ============================================================
# Step 5: Combine sub-location market value
# 80% sub-location market value
# 20% parent location desirability
# ============================================================

combined_score = (
    0.8 * df["global_percentile"] +
    0.2 * parent_norm
)


# ============================================================
# Step 6: Convert combined percentile to 1–5 score
# ============================================================

percentile = combined_score.rank(pct=True)

df["sub_location_score"] = pd.cut(
    percentile,
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
).astype(float)


# ============================================================
# Step 7: Handle Unspecified Sub-Locations
#
# If sub-location is:
#   Scheme 33_Unspecified
#   DHA Defence_Unspecified
#   Clifton_Unspecified
#
# use the parent location_score
# ============================================================

unknown_mask = (
    df["sub_location"]
    .str.endswith("_Unspecified", na=False)
)

df.loc[unknown_mask, "sub_location_score"] = (
    df.loc[unknown_mask, "location_score"]
)


# ============================================================
# Step 8: Safety fallback for any remaining NaN
# ============================================================

df["sub_location_score"] = (
    df["sub_location_score"]
    .fillna(df["location_score"])
)


# ============================================================
# Step 9: Convert to integer
# ============================================================

df["sub_location_score"] = (
    df["sub_location_score"]
    .astype(int)
)


# ============================================================
# Step 10: Remove helper column
# ============================================================

df.drop(
    columns=["global_percentile"],
    inplace=True
)


# ============================================================
# Step 11: Check result
# ============================================================

df.groupby("sub_location_score").agg(
    count=("sub_location", "count"),
    median_ppsqft=("price_per_sqft", "median")
)

,count,median_ppsqft
sub_location_score,,
1,3577,11111.110
2,3468,15277.780
3,4472,20283.215
4,4465,26666.670
5,4476,40000.000


In [69]:
df.groupby('sub_location_score')['price_crore'].agg(['count','mean','median'])

,count,mean,median
sub_location_score,,,
1,3577,1.992770,1.50
2,3468,3.122766,2.25
3,4472,4.761355,3.50
4,4465,8.075010,6.25
5,4476,15.428023,11.00


In [70]:
df.groupby(['sub_location_score', 'location', 'sub_location'])['price_per_sqft'].agg(
    count='count',
    median_ppsqft='median'
).reset_index()



,sub_location_score,location,sub_location,count,median_ppsqft
0,1,Bahria Town,Bahria Apartments,74,8176.100
1,1,Bahria Town,Bahria Heights,47,7103.830
2,1,Bahria Town,Bahria Paradise,56,10222.220
3,1,Bahria Town,Bahria Sports City,94,6349.210
4,1,Bahria Town,Precinct 10A,50,11111.110
5,1,Bahria Town,Precinct 10B,42,10133.330
6,1,Bahria Town,Precinct 11A,58,10789.470
7,1,Bahria Town,Precinct 11B,48,10633.045
8,1,Bahria Town,Precinct 12,85,12888.890
9,1,Bahria Town,Precinct 15,17,10488.890


In [71]:
df.head()

,property_name,address,location,sub_location,price,price_crore,price_per_sqft,area,bedrooms,bathrooms,...,property_type,property_age,age_category,parking_score,area_per_bedroom,district,sub_location_type,location_score,sub_location_score,expected_floors
0,"brand new, west open","saadi town - block 7, saadi town",Scheme 33,Sector 47,22800000.0,2.28,21111.11,1080.0,4,5,...,house,1.0,Relatively New,3,270.00,Karachi East,Subdivision,3,2,NaN
1,double storey house available for sale,"north nazimabad - block b, north nazimabad",North Nazimabad,North Nazimabad Block B,100000000.0,10.00,22222.22,4500.0,5,5,...,house,NaN,NaN,2,900.00,Karachi Central,Subdivision,4,2,2.0
2,166 square yard flat for sale in north nazimab...,"north nazimabad - block f, north nazimabad",North Nazimabad,North Nazimabad Block F,25500000.0,2.55,17068.27,1494.0,3,3,...,flat,NaN,NaN,0,498.00,Karachi Central,Subdivision,4,2,NaN
3,flat available for sale,"pechs block 2, pechs",Jamshed Town,Pechs,65000000.0,6.50,27049.52,2403.0,4,4,...,flat,NaN,NaN,1,600.75,Karachi East,Housing Society,4,4,NaN
4,north vista 2 bed d d flat for sale,"north nazimabad - block b, north nazimabad",North Nazimabad,North Nazimabad Block B,16500000.0,1.65,17295.60,954.0,2,3,...,flat,NaN,NaN,0,477.00,Karachi Central,Subdivision,4,2,NaN


In [72]:
df['property_age'].isnull().sum()

np.int64(9606)

In [73]:
df['district'].isnull().sum()

np.int64(385)

In [74]:
df[df['district'].isnull()]['location'].value_counts().index

Index(['Civil Lines', 'Jamshed Town', 'Saddar', 'Super Highway',
       'Gulshan-e-Iqbal Town', 'Shahra-e-Faisal', 'Northern Bypass', 'Malir'],
      dtype='object', name='location')

In [117]:
district_mapping = {
    # Karachi South
    "DHA Defence": "Karachi South",
    "Clifton": "Karachi South",

    # Karachi East
    "Karsaz": "Karachi East",
    "Gulshan-e-Iqbal Town": "Karachi East",
    "Gulistan-e-Jauhar": "Karachi East",
    "Jamshed Town": "Karachi East",
    "Scheme 33": "Karachi East",

    # Karachi Central
    "Federal B Area": "Karachi Central",
    "F.B. Area": "Karachi Central",
    "North Nazimabad": "Karachi Central",
    "North Karachi": "Karachi Central",
    "Nazimabad": "Karachi Central",
    "Naya Nazimabad": "Karachi Central",

    # Malir
    "Malir": "Malir",
    "Malir Cantt": "Malir",
    "Gadap": "Malir",
    "Super Highway": "Malir",
    "Bahria Town": "Malir",
    "Bin Qasim Town": "Malir",

    # Korangi
    "Korangi": "Korangi",
    "Shah Faisal": "Korangi"
}

In [76]:
df['district'].value_counts()

district
Karachi East       8077
Karachi South      5025
Malir              4126
Karachi Central    2533
Korangi             312
Name: count, dtype: int64

In [118]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20458 entries, 0 to 20478
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   property_name       20458 non-null  object 
 1   address             20458 non-null  object 
 2   location            20458 non-null  object 
 3   sub_location        20458 non-null  object 
 4   price               20458 non-null  float64
 5   price_crore         20458 non-null  float64
 6   price_per_sqft      20458 non-null  float64
 7   area                20458 non-null  float64
 8   bedrooms            20458 non-null  int64  
 9   bathrooms           20458 non-null  int64  
 10  kitchen             20458 non-null  float64
 11  floor               4008 non-null   float64
 12  total_floors        9609 non-null   float64
 13  parking             20458 non-null  float64
 14  elevator            20458 non-null  int64  
 15  servant_quarters    20458 non-null  float64
 16  electrici

In [78]:
df[df['sub_location_type'].isnull()]['sub_location'].value_counts()

sub_location
Scheme 33_Unspecified               302
Naya Nazimabad_Unspecified          301
Gulshan-e-Iqbal Town_Unspecified    188
Gulistan-e-Jauhar_Unspecified        96
Malir Cantt_Unspecified              95
North Nazimabad_Unspecified          80
Civil Lines_Unspecified              79
Saddar_Unspecified                   69
Federal B Area_Unspecified           58
Bahria Town_Unspecified              54
DHA City                             54
North Karachi_Unspecified            53
Clifton_Unspecified                  52
Defence View                         47
Sector 29                            43
Malir_Unspecified                    35
Shahra-e-Faisal_Unspecified          34
Tipu Sultan Road                     32
Phase 5                              28
DHA Defence_Unspecified              26
Northern Bypass_Unspecified          26
Naya Nazimabad Block C               24
Super Highway_Unspecified            23
Mateen Complex                       23
Naya Nazimabad Block A     

In [79]:
sub_location_type_mapping = {

    "Sector 29": "Subdivision",
    "Sector 40": "Subdivision",
    "Sector 35-A": "Subdivision",
    "Sector 19-A": "Subdivision",
    "Sector 24-A": "Subdivision",
    "Sector 17-A": "Subdivision",
    "Sector 25-B": "Subdivision",
    "Sector 15-A": "Subdivision",
    "Sector 18-A": "Subdivision",
    "Sector 42-A": "Subdivision",
    "Sector 44": "Subdivision",
    "Sector 30": "Subdivision",
    "sector 11a": "Subdivision",

    "Phase 2": "Subdivision",
    "Phase 5": "Subdivision",
    "Phase 6": "Subdivision",

    "Naya Nazimabad Block A": "Subdivision",
    "Naya Nazimabad Block B": "Subdivision",
    "Naya Nazimabad Block C": "Subdivision",
    "Naya Nazimabad Block D": "Subdivision",
    "Naya Nazimabad Block M": "Subdivision",

    "Gulshan-e-Iqbal Block 1": "Subdivision",
    "Gulshan-e-Iqbal Block 2": "Subdivision",
    "Gulshan-e-Iqbal Block 3": "Subdivision",
    "Gulshan-e-Iqbal Block 4": "Subdivision",
    "Gulshan-e-Iqbal Block 5": "Subdivision",
    "Gulshan-e-Iqbal Block 6": "Subdivision",
    "Gulshan-e-Iqbal Block 7": "Subdivision",
    "Gulshan-e-Iqbal Block 10": "Subdivision",
    "Gulshan-e-Iqbal Block 10A": "Subdivision",
    "Gulshan-e-Iqbal Block 11": "Subdivision",
    "Gulshan-e-Iqbal Block 13": "Subdivision",
    "Gulshan-e-Iqbal Block 13A": "Subdivision",
    "Gulshan-e-Iqbal Block 13D": "Subdivision",
    "Gulshan-e-Iqbal Block 13D2": "Subdivision",
    "Gulshan-e-Iqbal Block 19": "Subdivision",

    "Gulistan-e-Jauhar Block 1": "Subdivision",
    "Gulistan-e-Jauhar Block 2": "Subdivision",
    "Gulistan-e-Jauhar Block 3": "Subdivision",
    "Gulistan-e-Jauhar Block 3A": "Subdivision",
    "Gulistan-e-Jauhar Block 4": "Subdivision",
    "Gulistan-e-Jauhar Block 5": "Subdivision",
    "Gulistan-e-Jauhar Block 7": "Subdivision",
    "Gulistan-e-Jauhar Block 9": "Subdivision",
    "Gulistan-e-Jauhar Block 10": "Subdivision",
    "Gulistan-e-Jauhar Block 11": "Subdivision",
    "Gulistan-e-Jauhar Block 12": "Subdivision",
    "Gulistan-e-Jauhar Block 13": "Subdivision",
    "Gulistan-e-Jauhar Block 14": "Subdivision",
    "Gulistan-e-Jauhar Block 15": "Subdivision",
    "Gulistan-e-Jauhar Block 16": "Subdivision",
    "Gulistan-e-Jauhar Block 16A": "Subdivision",
    "Gulistan-e-Jauhar Block 17": "Subdivision",
    "Gulistan-e-Jauhar Block 18": "Subdivision",
    "Gulistan-e-Jauhar Block 19": "Subdivision",

    "Federal B Area Block 9": "Subdivision",
    "Federal B Area Block 11": "Subdivision",
    "Federal B Area Block 20": "Subdivision",
    "Federal B Area Block 21": "Subdivision",

    "Nazimabad 1": "Subdivision",
    "Nazimabad 2": "Subdivision",
    "Nazimabad 3": "Subdivision",
    "Nazimabad 4": "Subdivision",
    "Nazimabad 5": "Subdivision",

    "North Karachi Sector 5": "Subdivision",
    "North Karachi Sector 11": "Subdivision",

    "Precinct 1": "Subdivision",
    "Precinct 2": "Subdivision",
    "Precinct 4": "Subdivision",
    "Precinct 6": "Subdivision",
    "Precinct 8": "Subdivision",
    "Precinct 10A": "Subdivision",
    "Precinct 19": "Subdivision",

    "Punjabi Saudagar Society": "Housing Society",

    "Askari 5": "Military Housing",
    "Askari 6": "Military Housing",


    "Mateen Complex": "Residential Project",
    "Bahria Heights": "Residential Project",
    "Bahria Sports City": "Residential Project",
    "Falaknaz": "Residential Project",


    "Tipu Sultan Road": "Road",
    "Jamshed Road": "Road",
    "Shahra-e-Faisal": "Road",
    "Super Highway_Unspecified": "Unspecified",
    "Northern Bypass_Unspecified": "Unspecified",


    "Saadi Garden": "Neighbourhood",
    "Gulshan-E-Maymar": "Neighbourhood",
    "National Stadium Colony": "Neighbourhood",
    "Amil Colony": "Neighbourhood",
    "Model Colony": "Neighbourhood",
    "Bath Island": "Neighbourhood",
    "Airport": "Neighbourhood",
    "Jamshed Town_Unspecified": "Unspecified",
}

df["sub_location_type"] = df["sub_location"].map(sub_location_type_mapping).fillna(
    df["sub_location_type"]
)


In [80]:
df[df['sub_location_type'].isnull()]['sub_location'].value_counts()

sub_location
Scheme 33_Unspecified               302
Naya Nazimabad_Unspecified          301
Gulshan-e-Iqbal Town_Unspecified    188
Gulistan-e-Jauhar_Unspecified        96
Malir Cantt_Unspecified              95
North Nazimabad_Unspecified          80
Civil Lines_Unspecified              79
Saddar_Unspecified                   69
Federal B Area_Unspecified           58
DHA City                             54
Bahria Town_Unspecified              54
North Karachi_Unspecified            53
Clifton_Unspecified                  52
Defence View                         47
Malir_Unspecified                    35
Shahra-e-Faisal_Unspecified          34
DHA Defence_Unspecified              26
Shah Faisal_Unspecified              14
Nazimabad_Unspecified                 7
Sector 34-A                           4
Korangi_Unspecified                   3
Clifton Block 9                       2
Clifton Block 2                       2
Clifton Block 1                       1
Name: count, dtype: int64

In [81]:
sub_location_type_mapping = {

    # ---------------------------------------------------------
    # Unspecified parent-level locations
    # ---------------------------------------------------------
    "Scheme 33_Unspecified": "Unspecified",
    "Naya Nazimabad_Unspecified": "Unspecified",
    "Gulshan-e-Iqbal Town_Unspecified": "Unspecified",
    "Gulistan-e-Jauhar_Unspecified": "Unspecified",
    "Malir Cantt_Unspecified": "Unspecified",
    "North Nazimabad_Unspecified": "Unspecified",
    "Civil Lines_Unspecified": "Unspecified",
    "Saddar_Unspecified": "Unspecified",
    "Federal B Area_Unspecified": "Unspecified",
    "Bahria Town_Unspecified": "Unspecified",
    "North Karachi_Unspecified": "Unspecified",
    "Clifton_Unspecified": "Unspecified",
    "Malir_Unspecified": "Unspecified",
    "Shahra-e-Faisal_Unspecified": "Unspecified",
    "DHA Defence_Unspecified": "Unspecified",
    "Shah Faisal_Unspecified": "Unspecified",
    "Nazimabad_Unspecified": "Unspecified",
    "Korangi_Unspecified": "Unspecified",

    # ---------------------------------------------------------
    # DHA City
    # ---------------------------------------------------------
    "DHA City": "Housing Society",

    # ---------------------------------------------------------
    # Defence View
    # ---------------------------------------------------------
    "Defence View": "Housing Society",

    # ---------------------------------------------------------
    # Sectors
    # ---------------------------------------------------------
    "Sector 34-A": "Subdivision",

    # ---------------------------------------------------------
    # Clifton Blocks
    # ---------------------------------------------------------
    "Clifton Block 1": "Subdivision",
    "Clifton Block 2": "Subdivision",
    "Clifton Block 9": "Subdivision",
}
df["sub_location_type"] = df["sub_location"].map(sub_location_type_mapping).fillna(
    df["sub_location_type"]
)


In [82]:
df.head()

,property_name,address,location,sub_location,price,price_crore,price_per_sqft,area,bedrooms,bathrooms,...,property_type,property_age,age_category,parking_score,area_per_bedroom,district,sub_location_type,location_score,sub_location_score,expected_floors
0,"brand new, west open","saadi town - block 7, saadi town",Scheme 33,Sector 47,22800000.0,2.28,21111.11,1080.0,4,5,...,house,1.0,Relatively New,3,270.00,Karachi East,Subdivision,3,2,NaN
1,double storey house available for sale,"north nazimabad - block b, north nazimabad",North Nazimabad,North Nazimabad Block B,100000000.0,10.00,22222.22,4500.0,5,5,...,house,NaN,NaN,2,900.00,Karachi Central,Subdivision,4,2,2.0
2,166 square yard flat for sale in north nazimab...,"north nazimabad - block f, north nazimabad",North Nazimabad,North Nazimabad Block F,25500000.0,2.55,17068.27,1494.0,3,3,...,flat,NaN,NaN,0,498.00,Karachi Central,Subdivision,4,2,NaN
3,flat available for sale,"pechs block 2, pechs",Jamshed Town,Pechs,65000000.0,6.50,27049.52,2403.0,4,4,...,flat,NaN,NaN,1,600.75,Karachi East,Housing Society,4,4,NaN
4,north vista 2 bed d d flat for sale,"north nazimabad - block b, north nazimabad",North Nazimabad,North Nazimabad Block B,16500000.0,1.65,17295.60,954.0,2,3,...,flat,NaN,NaN,0,477.00,Karachi Central,Subdivision,4,2,NaN


In [83]:
floor_rows = df[
    df["property_name"]
    .astype(str).str.contains("floor", case=False, na=False)
]

floor_rows

,property_name,address,location,sub_location,price,price_crore,price_per_sqft,area,bedrooms,bathrooms,...,property_type,property_age,age_category,parking_score,area_per_bedroom,district,sub_location_type,location_score,sub_location_score,expected_floors
99,1100 sq feet 2-bed dd apartment for sale on 2n...,"dha phase 2 extension, dha defence",DHA Defence,Phase 2 Extension,14500000.0,1.450,13205.830000,1098.0,2,2,...,flat,NaN,NaN,1,549.000000,Karachi South,Subdivision,5,3,NaN
106,main alamgir road 1st floor apartment lift sta...,"alamgir road, gulshan-e-iqbal town",Gulshan-e-Iqbal Town,Bahadurabad,60000000.0,6.000,30030.030000,1998.0,3,3,...,flat,NaN,NaN,0,666.000000,Karachi East,Neighbourhood,4,4,NaN
117,saadi town block 7 extension 120 square yards...,"saadi town - block 7, saadi town",Scheme 33,Sector 47,24000000.0,2.400,22222.220000,1080.0,6,6,...,house,NaN,NaN,1,180.000000,Karachi East,Subdivision,3,2,2.0
126,luxurious 4 bed dd naz no-4 2nd floor,"nazimabad, karachi",Nazimabad,Nazimabad_Unspecified,33500000.0,3.350,14888.890000,2250.0,5,4,...,house,NaN,NaN,2,450.000000,Karachi Central,Unspecified,3,3,NaN
132,urgent house for sale in block 6 federal b are...,"federal b area - block 6, federal b area",Federal B Area,Federal B Area Block 6,60000000.0,6.000,31595.580000,1899.0,6,6,...,house,NaN,NaN,1,316.500000,Karachi Central,Subdivision,4,4,2.0
193,"ground rcc + 1st floor 2 room precast, 80 yard...","north karachi - sector 5-c/3, north karachi - ...",North Karachi,North Karachi Sector 5,12000000.0,1.200,16666.670000,720.0,4,3,...,house,NaN,NaN,2,180.000000,Karachi Central,Subdivision,2,2,NaN
198,apartment for sale dha phase 2 extension west ...,"dha phase 2 extension, dha defence",DHA Defence,Phase 2 Extension,15000000.0,1.500,13550.140000,1107.0,2,2,...,flat,NaN,NaN,0,553.500000,Karachi South,Subdivision,5,3,NaN
271,karachi university co operative housing societ...,"karachi university housing society, scheme 33",Scheme 33,Sector 18-A,13500000.0,1.350,10416.670000,1296.0,4,4,...,flat,NaN,NaN,0,324.000000,Karachi East,Subdivision,3,1,NaN
298,studio apartment for sale 4th floor bukhari co...,"bukhari commercial area, dha phase 6",DHA Defence,Phase 6,10000000.0,1.000,18214.940000,549.0,2,2,...,flat,NaN,NaN,0,274.500000,Karachi South,Subdivision,5,5,NaN
326,2nd floor 3 beds drawing dining flat,"gulshan-e-iqbal - block 13-d2, gulshan-e-iqbal",Gulshan-e-Iqbal Town,Gulshan-e-Iqbal Block 13D2,13000000.0,1.300,10860.480000,1197.0,3,3,...,flat,4.0,Relatively New,2,399.000000,Karachi East,Subdivision,4,2,NaN


In [84]:
floor_rows['property_name'].to_csv("floor_rows.csv", index=False)

In [85]:
import re



# ============================================================
# Extract floor number from property_name
# ============================================================

def extract_floor(text):

    if pd.isna(text):
        return np.nan

    text = str(text).lower().strip()

    # --------------------------------------------------------
    # 1. Ground floor
    # --------------------------------------------------------
    ground_patterns = [
        r'\bground\s+floor\b',
        r'\bgf\b',
        r'\bg\s+floor\b',
    ]

    for pattern in ground_patterns:
        if re.search(pattern, text):
            return 0

    # --------------------------------------------------------
    # 2. Written floor numbers
    # --------------------------------------------------------
    word_floors = {
        "first": 1,
        "second": 2,
        "third": 3,
        "fourth": 4,
        "fifth": 5,
        "sixth": 6,
        "seventh": 7,
        "eighth": 8,
        "ninth": 9,
        "tenth": 10,
        "eleventh": 11,
        "twelfth": 12,
        "thirteenth": 13,
        "fourteenth": 14,
        "fifteenth": 15,
        "sixteenth": 16,
        "seventeenth": 17,
        "eighteenth": 18,
        "nineteenth": 19,
        "twentieth": 20
    }

    for word, number in word_floors.items():

        pattern = rf'\b{word}\s+floor\b'

        if re.search(pattern, text):
            return number

    # --------------------------------------------------------
    # 3. Ordinal format
    #    1st floor, 2nd floor, 3rd floor, 4th floor...
    # --------------------------------------------------------
    match = re.search(
        r'\b(\d+)(?:st|nd|rd|th)\s+floor\b',
        text
    )

    if match:
        return int(match.group(1))

    # --------------------------------------------------------
    # 4. Simple numeric format
    #    1 floor, 2 floor, 3 floor...
    # --------------------------------------------------------
    match = re.search(
        r'\b(\d+)\s+floor\b',
        text
    )

    if match:
        return int(match.group(1))

    # --------------------------------------------------------
    # 5. No reliable floor found
    # --------------------------------------------------------
    return np.nan


# ============================================================
# Apply
# ============================================================

df["floor_from_name"] = df["property_name"].apply(extract_floor)

In [86]:
# Extract floor from property name
df["floor_from_name"] = df["property_name"].apply(extract_floor)

# Fill existing floor only where it is missing
df["floor"] = df["floor"].fillna(df["floor_from_name"])

# Optional: remove helper column
df.drop(columns=["floor_from_name"], inplace=True)

In [87]:
import re

def extract_total_floors(text):
    if pd.isna(text):
        return np.nan

    text = str(text).lower().strip()

    # --------------------------------------------------------
    # G+13, G + 13, G+3, etc.
    # G = ground floor (0), so total floors = number + 1
    # --------------------------------------------------------
    match = re.search(r'\bg\s*\+\s*(\d+)\b', text)

    if match:
        return int(match.group(1)) + 1

    # --------------------------------------------------------
    # Ground + 13, Ground + 3, etc.
    # --------------------------------------------------------
    match = re.search(r'\bground\s*\+\s*(\d+)\b', text)

    if match:
        return int(match.group(1)) + 1

    # --------------------------------------------------------
    # Ground and First / Ground, First and Second etc.
    # --------------------------------------------------------
    if re.search(r'\bground\b', text):

        floor_numbers = []

        word_numbers = {
            "first": 1,
            "second": 2,
            "third": 3,
            "fourth": 4,
            "fifth": 5,
            "sixth": 6,
            "seventh": 7,
            "eighth": 8,
            "ninth": 9,
            "tenth": 10
        }

        for word, number in word_numbers.items():
            if re.search(rf'\b{word}\b', text):
                floor_numbers.append(number)

        if floor_numbers:
            return max(floor_numbers) + 1

    return np.nan


# ============================================================
# Extract from property name
# ============================================================

df["total_floors_from_name"] = (
    df["property_name"]
      .apply(extract_total_floors)
)


# ============================================================
# Fill total_floors ONLY where it is missing
# ============================================================

df["total_floors"] = df["total_floors"].fillna(
    df["total_floors_from_name"]
)


# ============================================================
# Remove helper column
# ============================================================

df.drop(
    columns=["total_floors_from_name"],
    inplace=True
)

In [88]:
df['total_floors'].isnull().sum()

np.int64(10849)

In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20458 entries, 0 to 20478
Data columns (total 29 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   property_name       20458 non-null  object 
 1   address             20458 non-null  object 
 2   location            20458 non-null  object 
 3   sub_location        20458 non-null  object 
 4   price               20458 non-null  float64
 5   price_crore         20458 non-null  float64
 6   price_per_sqft      20458 non-null  float64
 7   area                20458 non-null  float64
 8   bedrooms            20458 non-null  int64  
 9   bathrooms           20458 non-null  int64  
 10  kitchen             20426 non-null  float64
 11  floor               4008 non-null   float64
 12  total_floors        9609 non-null   float64
 13  parking             20364 non-null  float64
 14  elevator            20458 non-null  int64  
 15  servant_quarters    20458 non-null  float64
 16  electrici

In [90]:
df[df['kitchen'].isnull()][['area','total_floors','price_crore']]

,area,total_floors,price_crore
1619,1377.0,NaN,2.050
2089,2160.0,3.0,11.000
2842,18000.0,NaN,40.000
2892,999.0,NaN,1.350
4123,1602.0,16.0,1.680
4178,405.0,NaN,0.265
5046,1350.0,3.0,3.950
5792,450.0,4.0,0.250
5957,1296.0,15.0,1.750
8325,999.0,4.0,1.950


In [91]:
df.groupby("kitchen").agg(
    count=("kitchen", "size"),
    mean_area=("area", "mean"),
    median_area=("area", "median"),
    min_area=("area", "min"),
    max_area=("area", "max"),
    mean_floors=("total_floors", "mean"),
    median_floors=("total_floors", "median"),
    min_floors=("total_floors", "min"),
    max_floors=("total_floors", "max")
).sort_index()

,count,mean_area,median_area,min_area,max_area,mean_floors,median_floors,min_floors,max_floors
kitchen,,,,,,,,,
0.0,4984,2133.768258,1503.0,252.0,27000.0,5.079853,2.0,1.0,49.0
1.0,10843,2171.237204,1746.0,225.0,36000.0,9.452729,5.0,1.0,92.0
2.0,3975,3115.510943,2250.0,198.0,36000.0,4.626730,2.0,1.0,48.0
3.0,576,3544.984375,2160.0,351.0,31500.0,2.955774,2.0,1.0,45.0
4.0,48,4956.750000,3600.0,648.0,30600.0,2.636364,3.0,1.0,4.0


In [92]:
kitchen_mapping = {
    1619: 1,
    2842: 4,
    2892: 1,
    4178: 1,
    9093: 4,
    9409: 3,
    10847: 2,
    11935: 1,
    14423: 1,
    15628: 3,
    16167: 2,
    17076: 2,
    19482: 2,
}

df.loc[df.index.isin(kitchen_mapping), "kitchen"] = (
    df.index.to_series().map(kitchen_mapping)
)

In [93]:
kitchen_mapping = {
    2089: 2,
    4123: 1,
    5046: 1,
    5792: 1,
    5957: 1,
    8325: 1,
    9222: 4,
    9388: 2,
    10570: 1,
    10852: 1,
    11873: 1,
    13159: 3,
    13640: 3,
    13701: 2,
    14196: 1,
    14720: 2,
    16306: 3,
    19361: 2,
    20175: 2
}

df.loc[kitchen_mapping.keys(), "kitchen"] = pd.Series(kitchen_mapping)

In [94]:
df['parking'].isnull().sum()

np.int64(94)

In [95]:
df.groupby("parking").agg(
    count=("parking", "size"),
    mean_area=("area", "mean"),
    median_area=("area", "median"),
    min_area=("area", "min"),
    max_area=("area", "max")
).sort_index()

,count,mean_area,median_area,min_area,max_area
parking,,,,,
0.0,6345,2095.231836,1503.0,252.0,27000.0
1.0,8561,2078.518981,1701.0,198.0,27000.0
2.0,2885,2528.046101,2250.0,342.0,21996.0
3.0,1104,3299.551630,3150.0,351.0,18000.0
4.0,843,3670.761566,3150.0,279.0,18000.0
5.0,314,4862.694268,3600.0,450.0,18000.0
6.0,134,6507.402985,6597.0,549.0,27000.0
7.0,29,6447.413793,9000.0,846.0,18000.0
8.0,60,6335.250000,4950.0,846.0,19800.0


In [96]:
df['parking'].isnull

<bound method Series.isnull of 0         3.0
1         1.0
2         0.0
3         2.0
4         0.0
5         4.0
6         1.0
7         4.0
8         1.0
9         0.0
10        3.0
11        0.0
12        1.0
13        2.0
14        2.0
15        1.0
16        0.0
17        1.0
18        0.0
19        1.0
20        0.0
21        0.0
22        1.0
23        1.0
24        2.0
25        0.0
26        1.0
27        2.0
28        1.0
29        0.0
30        1.0
31        1.0
32        0.0
33        1.0
34        1.0
35        0.0
36        1.0
37        2.0
38        1.0
39        1.0
40        1.0
41        1.0
42        0.0
43        2.0
44        5.0
45        0.0
46        1.0
47        0.0
48        1.0
49        1.0
50        0.0
51        1.0
52        1.0
53        1.0
54        1.0
55        0.0
56        2.0
57        2.0
58        1.0
59        1.0
60        0.0
61        5.0
62        2.0
63        4.0
64        1.0
65        1.0
66        2.0
67        0.0
68        0.0
69 

In [97]:
df["area_group"] = pd.cut(
    df["area"],
    bins=[0, 1200, 1800, 2400, 3600, 5000, 7000, 10000, float("inf")]
)

pd.crosstab(
    df["area_group"],
    df["parking"],
    normalize="index"
).round(3)

parking,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,14.0,15.0
area_group,,,,,,,,,,,,,,,
"(0.0, 1200.0]",0.398,0.446,0.088,0.025,0.026,0.007,0.003,0.000,0.002,0.001,0.004,0.0,0.000,0.000,0.000
"(1200.0, 1800.0]",0.323,0.498,0.110,0.031,0.023,0.008,0.002,0.000,0.002,0.000,0.002,0.0,0.000,0.000,0.000
"(1800.0, 2400.0]",0.277,0.468,0.165,0.047,0.030,0.009,0.002,0.000,0.001,0.000,0.001,0.0,0.000,0.000,0.000
"(2400.0, 3600.0]",0.245,0.377,0.251,0.060,0.041,0.016,0.004,0.002,0.001,0.000,0.002,0.0,0.000,0.000,0.000
"(3600.0, 5000.0]",0.211,0.291,0.204,0.185,0.080,0.017,0.008,0.002,0.003,0.000,0.001,0.0,0.000,0.000,0.000
"(5000.0, 7000.0]",0.281,0.274,0.078,0.126,0.173,0.048,0.007,0.002,0.007,0.002,0.002,0.0,0.000,0.000,0.000
"(7000.0, 10000.0]",0.237,0.203,0.042,0.055,0.153,0.136,0.097,0.024,0.034,0.002,0.013,0.0,0.000,0.000,0.003
"(10000.0, inf]",0.209,0.233,0.078,0.047,0.062,0.085,0.070,0.008,0.047,0.000,0.116,0.0,0.016,0.008,0.023


In [98]:
def estimate_parking(area):
    if pd.isna(area):
        return np.nan
    elif area < 1200:
        return 1
    elif area < 1800:
        return 1
    elif area < 2400:
        return 2
    elif area < 3600:
        return 2
    elif area < 5000:
        return 3
    elif area < 7000:
        return 4
    elif area < 10000:
        return 5
    else:
        return 5


df["parking"] = df["parking"].fillna(
    df["area"].apply(estimate_parking)
)

In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20458 entries, 0 to 20478
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   property_name       20458 non-null  object  
 1   address             20458 non-null  object  
 2   location            20458 non-null  object  
 3   sub_location        20458 non-null  object  
 4   price               20458 non-null  float64 
 5   price_crore         20458 non-null  float64 
 6   price_per_sqft      20458 non-null  float64 
 7   area                20458 non-null  float64 
 8   bedrooms            20458 non-null  int64   
 9   bathrooms           20458 non-null  int64   
 10  kitchen             20458 non-null  float64 
 11  floor               4008 non-null   float64 
 12  total_floors        9609 non-null   float64 
 13  parking             20458 non-null  float64 
 14  elevator            20458 non-null  int64   
 15  servant_quarters    20458 non-null  float

In [100]:
df["property_age_missing"] = df["property_age"].isna().astype(int)
df["total_floors_missing"] = df["total_floors"].isna().astype(int)

In [101]:
df.drop(columns=['expected_floors','area_group'],inplace=True)

In [102]:
df.head()

,property_name,address,location,sub_location,price,price_crore,price_per_sqft,area,bedrooms,bathrooms,...,property_age,age_category,parking_score,area_per_bedroom,district,sub_location_type,location_score,sub_location_score,property_age_missing,total_floors_missing
0,"brand new, west open","saadi town - block 7, saadi town",Scheme 33,Sector 47,22800000.0,2.28,21111.11,1080.0,4,5,...,1.0,Relatively New,3,270.00,Karachi East,Subdivision,3,2,0,0
1,double storey house available for sale,"north nazimabad - block b, north nazimabad",North Nazimabad,North Nazimabad Block B,100000000.0,10.00,22222.22,4500.0,5,5,...,NaN,NaN,2,900.00,Karachi Central,Subdivision,4,2,1,1
2,166 square yard flat for sale in north nazimab...,"north nazimabad - block f, north nazimabad",North Nazimabad,North Nazimabad Block F,25500000.0,2.55,17068.27,1494.0,3,3,...,NaN,NaN,0,498.00,Karachi Central,Subdivision,4,2,1,0
3,flat available for sale,"pechs block 2, pechs",Jamshed Town,Pechs,65000000.0,6.50,27049.52,2403.0,4,4,...,NaN,NaN,1,600.75,Karachi East,Housing Society,4,4,1,1
4,north vista 2 bed d d flat for sale,"north nazimabad - block b, north nazimabad",North Nazimabad,North Nazimabad Block B,16500000.0,1.65,17295.60,954.0,2,3,...,NaN,NaN,0,477.00,Karachi Central,Subdivision,4,2,1,1


In [103]:
df['property_type'].value_counts()

property_type
flat     10559
house     9899
Name: count, dtype: int64

In [114]:
df.head()

,property_name,address,location,sub_location,price,price_crore,price_per_sqft,area,bedrooms,bathrooms,...,completion_year,property_type,property_age,age_category,parking_score,area_per_bedroom,district,sub_location_type,location_score,sub_location_score
0,"brand new, west open","saadi town - block 7, saadi town",Scheme 33,Sector 47,22800000.0,2.28,21111.11,1080.0,4,5,...,2025.0,house,1.0,Relatively New,3,270.00,Karachi East,Subdivision,3,2
1,double storey house available for sale,"north nazimabad - block b, north nazimabad",North Nazimabad,North Nazimabad Block B,100000000.0,10.00,22222.22,4500.0,5,5,...,NaN,house,NaN,NaN,2,900.00,Karachi Central,Subdivision,4,2
2,166 square yard flat for sale in north nazimab...,"north nazimabad - block f, north nazimabad",North Nazimabad,North Nazimabad Block F,25500000.0,2.55,17068.27,1494.0,3,3,...,NaN,flat,NaN,NaN,0,498.00,Karachi Central,Subdivision,4,2
3,flat available for sale,"pechs block 2, pechs",Jamshed Town,Pechs,65000000.0,6.50,27049.52,2403.0,4,4,...,NaN,flat,NaN,NaN,1,600.75,Karachi East,Housing Society,4,4
4,north vista 2 bed d d flat for sale,"north nazimabad - block b, north nazimabad",North Nazimabad,North Nazimabad Block B,16500000.0,1.65,17295.60,954.0,2,3,...,NaN,flat,NaN,NaN,0,477.00,Karachi Central,Subdivision,4,2


In [113]:
df.drop(columns=['property_age_missing','total_floors_missing','age_category_filled','age_category_imputed_from'],inplace=True)

In [121]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20458 entries, 0 to 20478
Data columns (total 28 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   property_name       20458 non-null  object 
 1   address             20458 non-null  object 
 2   location            20458 non-null  object 
 3   sub_location        20458 non-null  object 
 4   price               20458 non-null  float64
 5   price_crore         20458 non-null  float64
 6   price_per_sqft      20458 non-null  float64
 7   area                20458 non-null  float64
 8   bedrooms            20458 non-null  int64  
 9   bathrooms           20458 non-null  int64  
 10  kitchen             20458 non-null  float64
 11  floor               4008 non-null   float64
 12  total_floors        9609 non-null   float64
 13  parking             20458 non-null  float64
 14  elevator            20458 non-null  int64  
 15  servant_quarters    20458 non-null  float64
 16  electrici

In [116]:
df[df['district'].isnull()]['location'].value_counts()

location
Civil Lines             79
Jamshed Town            79
Saddar                  69
Super Highway           54
Gulshan-e-Iqbal Town    37
Shahra-e-Faisal         31
Northern Bypass         26
Malir                   10
Name: count, dtype: int64

In [119]:
district_mapping = {
    # Karachi South
    "DHA Defence": "Karachi South",
    "Clifton": "Karachi South",
    "Civil Lines": "Karachi South",
    "Saddar": "Karachi South",

    # Karachi East
    "Karsaz": "Karachi East",
    "Gulshan-e-Iqbal Town": "Karachi East",
    "Gulistan-e-Jauhar": "Karachi East",
    "Jamshed Town": "Karachi East",
    "Scheme 33": "Karachi East",

    # Karachi Central
    "Federal B Area": "Karachi Central",
    "F.B. Area": "Karachi Central",
    "North Nazimabad": "Karachi Central",
    "North Karachi": "Karachi Central",
    "Nazimabad": "Karachi Central",
    "Naya Nazimabad": "Karachi Central",

    # Malir
    "Malir": "Malir",
    "Malir Cantt": "Malir",
    "Gadap": "Malir",
   
    "Bahria Town": "Malir",
    "Bin Qasim Town": "Malir",

    # Korangi
    "Korangi": "Korangi",
    "Shah Faisal": "Korangi",

    # Multi-District / Routes
    "Super Highway": "Multi-District",
    "Shahra-e-Faisal": "Multi-District",
    "Northern Bypass": "Multi-District"
}

In [120]:
df["district"] = df["location"].map(district_mapping)

In [122]:
df['age_category'].unique()

array(['Relatively New', nan, 'Old', 'New', 'Moderately Old',
       'Under Construction'], dtype=object)

In [129]:
df[df['total_floors'] < df['floor']]

,property_name,address,location,sub_location,price,price_crore,price_per_sqft,area,bedrooms,bathrooms,kitchen,floor,total_floors,parking,elevator,servant_quarters,electricity_backup,furnished,completion_year,property_type,property_age,age_category,parking_score,area_per_bedroom,district,sub_location_type,location_score,sub_location_score
14536,zamzam homes 3 bed drawing dining 2100 square ...,"bath island, karachi",Clifton,Bath Island,75000000.0,7.5,35765.38,2097.0,3,4,2.0,18.0,17.0,3.0,1,1.0,1,0,2025.0,flat,1.0,Relatively New,2,699.0,Karachi South,Neighbourhood,5,5


In [130]:
df.loc[14536, 'total_floors'] = 18

In [131]:
df.groupby('property_type')['floor'].apply(lambda x: x.isna().mean())

property_type
flat     0.626669
house    0.993333
Name: floor, dtype: float64

In [132]:
df.shape

(20458, 28)

In [133]:
df.to_csv('karachi_properties_handle_missing_value.csv', index=False)